# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule (CTR-Fix):** if a page ranks on the first page of google (Average Position <= 10) but has an abnormally low click-through rate (CTR < 3%) despite having solid visibility (Impressions > 1000), it gets flagged. This indicates the page is highly visible to users, but the meta title or snippet is failing to win the click.

**Reasin Code:** LOW_CTR_HIGH_VISIBILITY

**Action Label:** UPDAYE_META_TAGS

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import os

# Safely load the anonymized dataset (Handles both local repo and virtual Colab environments)
path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(path):
    path = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(path)

# Dynamically map columns to prevent KeyErrors
pos_col = [c for c in df.columns if 'position' in c.lower()][0]
ctr_col = [c for c in df.columns if 'ctr' in c.lower()][0]
imp_col = [c for c in df.columns if 'impression' in c.lower()][0]

print("--- SIGNAL 1: Position vs CTR (CTR-Fix Logic) ---")
df['position_bucket'] = pd.qcut(df[pos_col], q=4, duplicates='drop')
signal_1 = df.groupby('position_bucket', observed=True)[ctr_col].agg(['mean', 'count']).rename(columns={'count': 'n'})
display(signal_1)
print("Verdict: CONFIRMED (Higher ranking positions strongly correlate with higher CTRs. A low CTR at a high position is a confirmed anomaly).\n")

print("--- SIGNAL 2: Volume vs Position (Quick-Win Logic) ---")
df['impression_bucket'] = pd.qcut(df[imp_col], q=4, duplicates='drop')
signal_2 = df.groupby('impression_bucket', observed=True)[pos_col].agg(['mean', 'count']).rename(columns={'count': 'n'})
display(signal_2)
print("Verdict: MIXED (Massive impressions do not guarantee page 1 rankings. High volume pages stuck on page 2 are prime 'quick-win' targets).")

--- SIGNAL 1: Position vs CTR (CTR-Fix Logic) ---


,mean,n
position_bucket,,
"(-0.001, 6.2]",1.079626,7543
"(6.2, 10.8]",0.436351,7534
"(10.8, 22.3]",0.319024,7462
"(22.3, 245.0]",0.202433,7461


Verdict: CONFIRMED (Higher ranking positions strongly correlate with higher CTRs. A low CTR at a high position is a confirmed anomaly).

--- SIGNAL 2: Volume vs Position (Quick-Win Logic) ---


,mean,n
impression_bucket,,
"(0.999, 81.0]",13.279328,7503
"(81.0, 731.0]",21.491306,7499
"(731.0, 3615.25]",17.112137,7498
"(3615.25, 517715.0]",13.488867,7500


Verdict: MIXED (Massive impressions do not guarantee page 1 rankings. High volume pages stuck on page 2 are prime 'quick-win' targets).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

id_col = 'content_hash_id' if 'content_hash_id' in df.columns else df.columns[0]

# Applying the Baseline Rule (CTR-Fix)
mask = (df[pos_col] <= 10) & (df[ctr_col] < 0.03) & (df[imp_col] > 1000)
baseline_df = df[mask].copy()

baseline_df['action_label'] = 'UPDATE+META_TAGS'
baseline_df['reason_code'] = 'LOW_CTR_HIGH_VISIBILITY'

# Now we score by impressions to prioritize fixing pages that will yield the most recovered traffic
baseline_df['baseline_score'] = baseline_df[imp_col]

# Ranking the queue (Highest risk/score first)
ranked_queue = baseline_df.sort_values(by='baseline_score', ascending=False)

# Saving to work/outputs/ (Handle missing directories safely in Colab)
out_dir = '../work/outputs' if os.path.exists('../work') else 'work/outputs'
os.makedirs(out_dir, exist_ok=True)
ranked_queue[[id_col, 'action_label', 'reason_code', 'baseline_score', imp_col, pos_col, ctr_col]].to_csv(f'{out_dir}/baseline_action_score.csv', index=False)

print(f"Queue generated! {len(ranked_queue)} pages flagged.")
display(ranked_queue.head(20))

Queue generated! 326 pages flagged.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,position_bucket,impression_bucket,action_label,reason_code,baseline_score
7445,content_c8e9d6ab9013,client_19581e27de,20.0,0.03,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,excellent,page_1,down,-43.4,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,208678
27178,content_453722754fea,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2700.0,18723.0,...,0.0,excellent,page_1,down,-52.9,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,140079
3331,content_4a6607efcb46,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4939.0,32266.0,...,2.3,excellent,top_3,up,5426.6,"(-0.001, 6.2]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,128068
15914,content_0919dd345d80,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,2841.0,18175.0,...,0.0,excellent,page_1,down,-76.1,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,119217
482,content_39881853ef0c,client_f369cb89fc,170.0,0.02,LOW,0.00,keyword article,informational,2810.0,19393.0,...,0.0,excellent,page_1,down,-42.0,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,112434
2382,content_65114d89496d,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.0,excellent,page_1,down,-74.8,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,72631
14226,content_65d9331f55fb,client_4e07408562,50.0,0.00,LOW,0.00,keyword article,informational,3161.0,20147.0,...,0.0,excellent,page_1,down,-79.3,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,65686
13631,content_d274ac4158ef,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,1428.0,8944.0,...,0.0,excellent,page_1,stable,-11.8,"(6.2, 10.8]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,65138
24866,content_e5f459e737b7,client_f369cb89fc,30.0,1.00,HIGH,3.90,keyword article,transactional,2916.0,17229.0,...,0.0,excellent,page_1,down,-27.2,"(-0.001, 6.2]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,56363
4589,content_339b357d04c7,client_bbb965ab0c,0.0,0.00,LOW,0.00,keyword article,informational,2484.0,18526.0,...,0.0,excellent,page_1,up,865.1,"(-0.001, 6.2]","(3615.25, 517715.0]",UPDATE+META_TAGS,LOW_CTR_HIGH_VISIBILITY,46879


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top 1-5:** Massive impression volume (>10k), Page 1 ranking, but <1% CTR. Clearly suffering from poor meta descriptions. Action: Update Meta. Wrong if: The search query is highly informational and perfectly answered by a Google Featured Snippet (a "zero-click" search), meaning nobody clicks the link regardless of the title.

**Top 6-15:** Solid volume, but sitting near the bottom of page 1 (Position 8-10). Action: Update Meta. Wrong if: We are applying a flat 3% CTR threshold. A 2% CTR is terrible for Position 1, but might actually be standard/healthy for Position 9.

**Top 16-20:** Borderline cases with impressions hovering just above the 1,000 cutoff. Action: Update Meta. Wrong if: This is a seasonal keyword where impressions spiked briefly, but user intent doesn't actually align with the page content.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:** The baseline rule treats CTR linearly. Flagging position 9 pages with a flat <3% CTR threshold generates weak picks because click curves drop exponentially at the bottom of page 1. Our future ML model will need to learn the non-linear relationship between rank and expected CTR.

**Leakage Check:** Confirmed clean. We are exclusively using raw, historical Search Console metrics. No future outcome labels, downstream traffic data, or proprietary FlyRank product flags were leaked into the rule logic.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.